# 01 · Transcutaneous Electrical Stimulation

**Physics of Electrical Neurostimulation** · Taller Escuela de Neurociencia (FALAN School), Santiago, August 2026

Leonel Medina · Rodrigo Osorio · Cristian Morales — [NeuroEng@USACH](https://www.neuroeng-usach.cl), Universidad de Santiago de Chile

| | |
|---|---|
| Notebook 01 | **Transcutaneous stimulation** — the field, the axon, the strength-duration curve |
| Notebook 02 | **Interferential current** — two carriers, one beat, deep activation |
| Lab | TENS/EMS unit on your own forearm; handouts in `handouts/` |
| No install | `explorer/index.html` runs the two core figures in any browser |

### What you will be able to do by the end
1. Compute and read the electric field of a surface electrode pair over layered tissue, and say which feature of that field excites a nerve.
2. Predict where an axon fires from the activating function, then check that prediction against a real nonlinear membrane simulation.
3. Measure a strength-duration curve on your own forearm, fit rheobase and chronaxie, and compare all three: your data, this model, and the published human value.
4. Explain why cathodic and anodic thresholds differ — and the conditions under which they don't.

### How the 80 minutes are spent
| | |
|---|---|
| Module 0 — the electric field itself | ~20 min |
| Module 1 — one stimulus: does the fibre fire? | ~20 min |
| Module 2 — strength-duration curve, model vs. your forearm | ~30 min (simulation runs while you measure) |
| Wrap-up and hand over to Notebook 02 | ~10 min |

> **Start the Module 2 simulation early.** A full five-point sweep takes about a minute of real
> computation. Launch it, then go and put electrodes on someone's forearm while it runs.

## 0 · Setup

One cell, no editing. On Colab it also installs NEURON and PyFibers (about a minute).

In [1]:
# One-time setup: fetches the setup script if needed and runs it. On Colab it also
# installs NEURON, which requires the kernel to restart once -- if that happens,
# just run this cell again. Curious what it does? Open notebooks/setup_workshop.py
import pathlib, urllib.request
URL = ("https://raw.githubusercontent.com/neuroeng-usach/"
       "falan-neurostim-workshop/main/notebooks/setup_workshop.py")
if not pathlib.Path("setup_workshop.py").exists():
    urllib.request.urlretrieve(URL, "setup_workshop.py")
%run -i setup_workshop.py

RESTART REQUIRED — numpy changed from 2.3.3 to 2.2.6 while this kernel was running.
  Colab:  Runtime -> Restart session,  then run this cell again.
  Local:  restart the kernel, then run this cell again.
  Nothing below will work until you do; this is not an error in the
  notebook, it is how compiled Python packages behave.
repo root: /Users/leo/Library/CloudStorage/GoogleDrive-leonel.medina@usach.cl/My Drive/Outreach/FALAN School/release
[self-test] current conservation: integrated 0.9960 mA vs injected 1.0000 mA (ratio 0.9960) -- PASS
[self-test] all analytic sanity checks passed:
  continuity at boundary: 89.665828 vs 89.665778 mV
  homogeneous-limit match: 94.901672 vs 94.901672 mV
  insulating backing raises V: 807.8492 > 94.9017
  conductive backing lowers V: 10.1876 < 94.9017
[self-test] disc electrode checks passed:
  tiny-disc -> point-source convergence: rel diff 3.49e-06
  peak V monotonically decreasing with radius: ['3735.4', '2216.7', '889.5', '500.1', '272.0', '153.7']

--No graphics will be displayed.


ENVIRONMENT READY — field figures and axon simulations will both run.


True

## What is actually being computed here

Three modules, all in Python, no FEM license needed, and no hidden one-line formulas.

- **Module 0 — the electric field itself.** A two-layer volume conductor (skin+fat over muscle),
  derived from first principles by the method of images and validated against current
  conservation, in `src/layered_field.py`. The electrode can be a point source or a realistic
  **disc** of adjustable radius (a superposition of point sources), and the field is driven by a
  **bipolar pad pair** — source and sink, adjustable separation — matching how the TENS unit you
  will actually test with works.
- **Module 1 — cathodic vs. anodic, one stimulus at a time.** The *same* field from Module 0 now
  drives a real myelinated axon (MRG double-cable model, PyFibers/NEURON).
- **Module 2 — strength-duration curve.** Real bisection threshold searches on the nonlinear
  membrane, fitted for rheobase and chronaxie, compared against your own forearm measurements.

Every slider you move in Module 0 carries forward: Modules 1 and 2 inherit the tissue
parameters you set there, through the shared store `ws.STATE`. Print it any time to see the
full parameter set you are working with.

*No sliders in your environment?* Nothing breaks. Each interactive cell falls back to running
once with its defaults and tells you how to override them by hand — there is no separate
"fallback" notebook to keep track of.

## Module 0 — The electric field of a transcutaneous electrode pair

**Before you run it:** with the cathode (negative current) on and the muscle
layer *more* conductive than the shallow layer (the default), do you expect
the deep layer to concentrate current toward it, or spread it out? Run and check.

**Then try this:** raise the electrode radius from 0 (a point) to a realistic
disc pad (e.g. 5-10 mm, roughly a small TENS/EMS electrode). Does the peak
current density under the electrode go up or down? Does the activating
function at the fiber get sharper or more spread out?

**This is bipolar by default** -- two electrodes (a source and a sink)
`separation_mm` apart, exactly like the two pads on the TENS unit you'll
test with, instead of one electrode with an implicit distant return. Try
changing the separation: what happens to the field between the two pads
as they get closer together vs. farther apart?

### Mathematical Background: Method of Images

To model the electric field in a two-layer tissue (e.g., skin/fat over muscle), we cannot simply use the equation for a homogeneous medium. The boundary between the two tissues reflects the electric field, governed by the **reflection coefficient** $k$:

$$ k = \frac{\sigma_1 - \sigma_2}{\sigma_1 + \sigma_2} $$

where $\sigma_1$ and $\sigma_2$ are the conductivities of the first (shallow) and second (deep) layers.

Using the **Method of Images**, the potential field $V$ is calculated as an infinite series reflecting between the skin-air interface (which is a perfect insulator, reflecting 100% of the current) and the skin-muscle interface:

**Potential in Layer 1 (Skin/Fat):**
$$ V_1(r, z) = \frac{I_0}{2 \pi \sigma_1} \left[ \frac{1}{\sqrt{r^2 + z^2}} + \sum_{n=1}^{\infty} k^n \left( \frac{1}{\sqrt{r^2 + (z - 2nh)^2}} + \frac{1}{\sqrt{r^2 + (z + 2nh)^2}} \right) \right] $$

**Potential in Layer 2 (Muscle):**
$$ V_2(r, z) = \frac{I_0}{\pi (\sigma_1 + \sigma_2)} \sum_{n=0}^{\infty} \frac{k^n}{\sqrt{r^2 + (2nh - z)^2}} $$

This robust approach ensures current conservation across the boundary, providing a realistic picture of how current spreads.

**From a point to a disc electrode.** The formulas above are for an idealized point contact. A real transcutaneous electrode has a finite area, and a disc electrode of radius $a$ carrying uniform current density is modeled here as a superposition of many point sources, each injecting $I_0/N$, spread uniformly over the disc:

$$ V_{disc}(r, z) = \sum_{i=1}^{N} V_{point}\!\left(|\mathbf{r} - \mathbf{r}_i|, z; \frac{I_0}{N}\right) $$

This is the standard boundary-element-style approximation for a finite contact: as $N \to \infty$ it converges to the exact uniform-current-density disc solution, and it lets us reuse the exact two-layer point-source formulas above unchanged. In the interactive visualization below, you will see a comparison of the activating function using this realistic two-layer model versus a simple homogeneous model, for whichever electrode geometry you choose.

**A caveat worth knowing:** this assumes *uniform* current density across the disc. A real gelled electrode with a metal contact behaves more like a constant-potential surface instead, which pushes current density higher at the pad's edges than its center (the classic disk-electrode edge effect). The model above should get the *direction* of electrode-size effects right -- larger pad, lower peak, broader activation -- but not necessarily their exact magnitude.

**From one electrode to a pad pair (bipolar).** A real bipolar device (TENS included) has no separate ground -- current flows from one pad to the other. Because the governing equation is linear in injected current, this is exact by superposition: place a source ($+I_0$) at one pad and a sink ($-I_0$) at the other, `separation_mm` apart, and add their fields:

$$ V_{bipolar}(x, z) = V(x + \tfrac{s}{2}, z; +I_0) + V(x - \tfrac{s}{2}, z; -I_0) $$

This is `layered_field.bipolar_potential()`, used by Module 0's field plot below.

### From field to excitation: the Activating Function

The field above is a property of the *tissue* -- it exists whether or not a nerve is there. What actually excites a nerve fiber sitting in that field? According to the **cable equation**, the extracellular potential $V_e$ does not directly drive the membrane. Instead, it is the *second spatial derivative* of $V_e$ along the fiber that acts as the driving source for membrane polarization -- the **Activating Function** $f(x)$:

$$ f(x) = \frac{\partial^2 V_e}{\partial x^2} $$

- Where $f(x) > 0$ (positive), the field acts to **depolarize** the membrane, pushing it toward the firing threshold.
- Where $f(x) < 0$ (negative), the field acts to **hyperpolarize** the membrane, suppressing firing.

This is why the field plot below is paired with an activating-function plot, not just a potential map -- the *shape* of $V_e$ (how sharply it curves), not just its magnitude, is what determines whether and where a fiber would fire. Module 1 takes this one step further: is this linear prediction actually where a real, nonlinear nerve fiber fires?

In [ ]:
panel0 = ws.Panel(
    tp.draw_module0,
    controls=[
        ws.num("sigma1",              "sigma1 skin+fat (S/m)",        0.01, 0.30, 0.01, lf.DEFAULT_SIGMA1),
        ws.num("sigma2",              "sigma2 muscle (S/m)",          0.05, 1.00, 0.01, lf.DEFAULT_SIGMA2),
        ws.num("h_mm",                "shallow layer thickness (mm)", 1.0, 12.0,  0.5,  lf.DEFAULT_H_MM),
        ws.num("i0_mA",               "current (mA, neg = cathodic)", -10.0, 10.0, 0.1, -2.0),
        ws.num("fiber_depth_mm",      "fibre depth (mm)",             1.0, 20.0,  0.5,  8.0),
        ws.num("electrode_radius_mm", "pad radius (mm, 0 = point)",   0.0, 15.0,  0.5,  4.0),
        ws.num("separation_mm",       "pad separation (mm)",          20.0, 120.0, 5.0, 60.0),
    ],
    button="Draw field",
).show()

## Module 1 — One stimulus: does the fiber fire? (cathodic vs. anodic)

Same field as Module 0, now driving a real fiber. Pick an amplitude, pulse
width, and polarity, then run -- the top panel shows the activating-function
prediction from Module 0's theory (dotted line) next to where the real,
nonlinear NEURON simulation actually initiates the spike, if it fires
(dashed line).

**Electrode mode defaults to monopolar** (single electrode, implicit distant
return) -- this is what produces the classic, literature-matching
cathodic/anodic threshold asymmetry (~3.9x) you'll read about below.

**Try switching to bipolar mode with the default (symmetric) settings.**
What happens to the cathodic/anodic difference? Think about *why* before
reading the facilitator notes -- it's a real, physically meaningful result
(not a bug), and it says something important about what "reversing
polarity" actually means once there's a real return pad instead of a
distant implicit one. Bipolar mode also needs a longer simulated fiber and
runs noticeably slower (~15-25s per threshold vs. ~5-10s monopolar) --
that's expected, not a freeze.

### From prediction to test: why run a real simulation at all?

The activating function (Module 0) is a *linear* approximation -- it assumes the membrane responds proportionally to the driving field, which only holds for small perturbations away from rest. Whether the fiber actually fires a propagating action potential depends on the full *nonlinear* membrane dynamics (voltage-gated sodium/potassium channels -- the same Hodgkin-Huxley-family kinetics built into the MRG model here), which is exactly what PyFibers/NEURON simulates directly, compartment by compartment, with no shortcuts.

The top panel below shows both: the **linear prediction** (peak of the activating function, dotted line) and, if the fiber fires, **where the nonlinear simulation actually initiates the action potential** (dashed line -- the earliest node to cross threshold). Near threshold amplitude they should agree closely, since the linear approximation is most accurate right where the membrane is just barely being pushed over the edge -- that's exactly the regime it was derived for.

**Try this:** run near threshold first, then crank the amplitude well past it. Does the actual initiation site stay locked to the prediction, or drift as the response becomes more nonlinear?

In [ ]:
# Tissue geometry is inherited from Module 0 -- change it there and it changes here.
panel1 = ws.Panel(
    tp.draw_module1,
    controls=[
        ws.choice("diameter",       "fibre diameter (um)",
                  [5.7, 7.3, 8.7, 10.0, 11.5, 12.8, 14.0, 15.0, 16.0], 10.0),
        ws.num("amplitude_mA",     "amplitude (mA)",    0.05, 20.0, 0.05, 3.0),
        ws.num("pulse_width_ms",   "pulse width (ms)",  0.02,  2.0, 0.01, 0.3),
        ws.choice("polarity",      "polarity", ["cathodic", "anodic"], "cathodic"),
        ws.choice("electrode_mode", "electrode mode",
                  [("monopolar (single electrode)", "monopolar"),
                   ("bipolar (pad pair, like the real TENS)", "bipolar")], "monopolar"),
    ],
    inherit=["fiber_depth_mm", "sigma1", "sigma2", "h_mm",
             "electrode_radius_mm", "separation_mm"],
    button="Run simulation",
    note="Bipolar mode uses a longer fibre and takes ~15-25 s per run; monopolar ~5-10 s.",
).show()

## Module 2 — Strength-duration curve

Computes threshold at several pulse widths (real nonlinear simulations — can
take up to ~1-2 minutes, longer in bipolar mode), fits rheobase & chronaxie,
and compares to the published human ulnar-nerve values. Then compare to your
own forearm data.

Uses the same electrode mode / pad separation as Module 1 above (the
`mode1_dd` / `sep1_sl` widgets) -- change those first if you want a bipolar
sweep here.

### Which pulse widths to sweep

The model is happy over 50 us - 1 ms. Your **TENS Ultima Neo in monophasic rectangular mode is
limited to 50-250 us**, so the two ranges only partly overlap. Sweep the device-matched set if
you want a like-for-like comparison, or the wider set to see the whole curve bend. Doing both and
discussing the mismatch is the honest option — and it is the range where chronaxie actually lives.

Set `MY_DATA` to whatever you measured on the forearm (one `pulse_width_ms, threshold_mA` pair per
line; paste straight from the handout table, comments and all — non-numeric lines are ignored).

In [ ]:
# --- what to sweep ----------------------------------------------------------
PULSE_WIDTHS_MS = (0.05, 0.10, 0.20, 0.50, 1.00)      # full model range
# PULSE_WIDTHS_MS = (0.05, 0.10, 0.15, 0.20, 0.25)    # Ultima Neo monophasic range
if ws.FAST:                                            # smoke test only (WORKSHOP_FAST=1)
    PULSE_WIDTHS_MS = PULSE_WIDTHS_MS[::2]

# --- your forearm measurements (pulse_width_ms, motor threshold mA) ---------
MY_DATA = """
0.05,
0.10,
0.15,
0.20,
0.25,
"""

sim = tp.compute_module2(
    diameter=ws.STATE["diameter"], fiber_depth_mm=ws.STATE["fiber_depth_mm"],
    sigma1=ws.STATE["sigma1"], sigma2=ws.STATE["sigma2"], h_mm=ws.STATE["h_mm"],
    polarity=ws.STATE["polarity"], pulse_widths_ms=PULSE_WIDTHS_MS,
    electrode_radius_mm=ws.STATE["electrode_radius_mm"],
    electrode_mode=ws.STATE["electrode_mode"], separation_mm=ws.STATE["separation_mm"],
)
pts, fit = sim

my_pts = ws.parse_pairs(MY_DATA)
tp.plot_module2(pts, fit, my_pts or None, weiss.fit_weiss(my_pts) if len(my_pts) > 1 else None)

## Bridge to the lab — measuring this on a forearm

The number this notebook predicts is a **motor threshold as a function of pulse width**, so the
measurement has to be a motor threshold too. The full protocol is in
`handouts/01_transcutaneous_handout.md`; the four things that decide whether your curve has any
shape at all:

1. **Single pulses, not trains.** At 30-50 Hz the contraction appears through mechanical fusion
   and temporal summation, and that threshold is almost completely insensitive to pulse width —
   a flat "curve" is the guaranteed result. Use the lowest frequency the device offers (1-2 Hz).
2. **Monophasic mode.** Symmetric biphasic pulses reverse polarity within each pulse, which both
   erases the cathodic/anodic comparison of Module 1 and compresses the long-pulse-width end of
   the curve. Confirm which lead is the cathode before you start — do not trust a red/black
   convention.
3. **A visible, repeatable endpoint.** Median nerve at the wrist, watching thumb abduction, with a
   light pointer taped to the thumb against a ruler. Threshold = the lowest amplitude giving a
   visible response in **5 of 10 pulses**.
4. **Randomised pulse-width order, and a blind scorer.** Then repeat the first condition at the
   end: skin impedance drifts downward over a session by about as much as the effect you are
   trying to measure.

Your instructors have bench-tested the unit you are using for the traps that produce a
plausible-looking but meaningless curve: output-compliance clipping, trains where you expected
single pulses, and which physical lead is actually the cathode. Ask them what they found before you
start — the answers should be on the whiteboard.

## What to compare, and what it means

| Quantity | Your forearm | This model | Published (near-nerve ulnar) |
|---|---|---|---|
| Rheobase | | | 0.91 mA (SD 0.37) |
| Chronaxie | | | 0.32 ms (SD 0.17) |

**Chronaxie is the robust prediction.** It mostly reflects membrane kinetics, so the model lands
near the published value almost regardless of the depth and conductivity you chose.
**Rheobase is the sensitive one** — it scales with how far the electrode sits from the nerve and
with the tissue conductivities. If your measured rheobase does not match, go back to Module 0 and
adjust `sigma1`, `sigma2` and `fiber_depth_mm` until it does, then ask what that implies about
where the nerve actually is under your pads. A transcutaneous surface measurement sits farther
from the nerve than Tsui et al.'s near-nerve needle, so a **higher** rheobase is the expected result, not an
error.

### Next: Notebook 02
Everything so far used one low-frequency pulse through one pad pair, and its reach into depth was
set by pad separation. Notebook 02 asks the opposite question: can you deliver a low-frequency
stimulus *deep* while keeping the skin comfortable, by generating the low frequency **inside** the
tissue instead of applying it at the surface? Same volume conductor, four pads.